# 11 — LSQ Superpixel Glint Correction

Copy of `10_glint_correction_lsq.ipynb` with the per-pixel LSQ inversion replaced by the
SLIC superpixel pipeline from `superpixel_engine.py`.

Pipeline:
1. **Segment** the Rrs image into spectrally coherent superpixels (SLIC)
2. **Invert** the mean spectrum of each superpixel with pure LSQ (no prior)
3. **Back-interpolate** to full pixel resolution via PCA + kNN IDW

The superpixel approach inverts ~N_SEGMENTS spectra instead of all pixels,
giving a large speedup while preserving spatial detail through the back-interpolation.

**Chi-2 calibration note (LSQ + superpixels):**  
`invert_superpixels` passes the pixel-level noise unchanged.  Mean spectra are
smoother (variance σ²/N), so `chi2_raw` ≈ chi2_per_pixel / N.  
The calibrated chi2 stored in the output is `chi2_calibrated = chi2_raw × N`;
its ideal value is `(n_obs − n_fit) / n_obs` — same as for per-pixel LSQ.

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import scipy.ndimage as ndi
import lmfit
import time
import xarray as xr
import rioxarray
import jax
import jax.numpy as jnp
from pyproj import CRS
from xcube.core.store import new_data_store
import configparser
from skimage.segmentation import mark_boundaries

from bio_optics.coupled_models import albert_mobley_3C_jax
from bio_optics.inversion import oe_engine
from bio_optics.inversion import lsq_engine_optx as lsq_engine
from bio_optics.inversion import superpixel_engine
from bio_optics.surface import air_water
from bio_optics.surface.reflectance import Rrs_surf
from bio_optics.atmosphere import sky_radiance

jax.config.update('jax_enable_x64', True)

## Parameters and forward model

Identical to NB10.  No `sigma_a` — pure least-squares.

In [ ]:
params_3C = lmfit.Parameters()
params_3C.add('C_0',   value=2,          min=0,     max=100, vary=True)
params_3C.add('C_1',   value=0,          min=0,     max=100, vary=False)
params_3C.add('C_2',   value=0,          min=0,     max=100, vary=False)
params_3C.add('C_3',   value=0,          min=0,     max=100, vary=False)
params_3C.add('C_4',   value=0,          min=0,     max=100, vary=False)
params_3C.add('C_5',   value=0,          min=0,     max=100, vary=False)
params_3C.add('C_Y',   value=0.2,        min=0,     max=4,   vary=True)
params_3C.add('C_X',   value=5,          min=0,     max=100, vary=True)
params_3C.add('C_Mie', value=1,          min=0,     max=100, vary=True)
params_3C.add('f_0',   value=0,          min=0,     max=1,   vary=False)
params_3C.add('f_1',   value=1,          min=0,     max=1,   vary=False)
params_3C.add('f_2',   value=0,          min=0,     max=1,   vary=False)
params_3C.add('f_3',   value=0,          min=0,     max=1,   vary=False)
params_3C.add('f_4',   value=0,          min=0,     max=1,   vary=False)
params_3C.add('f_5',   value=0,          min=0,     max=1,   vary=False)
params_3C.add('B_0',   value=1/np.pi,                        vary=False)
params_3C.add('B_1',   value=1/np.pi,                        vary=False)
params_3C.add('B_2',   value=1/np.pi,                        vary=False)
params_3C.add('B_3',   value=1/np.pi,                        vary=False)
params_3C.add('B_4',   value=1/np.pi,                        vary=False)
params_3C.add('B_5',   value=1/np.pi,                        vary=False)
params_3C.add('bb_phy_spec',         value=0.0010,            vary=False)
params_3C.add('bb_Mie_spec',         value=0.0042,            vary=False)
params_3C.add('bb_X_spec',           value=0.0086,            vary=False)
params_3C.add('a_NAP_spec_lambda_0', value=0.041,             vary=False)
params_3C.add('S',                   value=0.014,             vary=False)
params_3C.add('K',                   value=0,                 vary=False)
params_3C.add('S_NAP',               value=0.011,             vary=False)
params_3C.add('n',                   value=-1,                vary=False)
params_3C.add('lambda_0',            value=440,               vary=False)
params_3C.add('lambda_S',            value=500,               vary=False)
params_3C.add('theta_sun',  value=np.radians(30),             vary=False)
params_3C.add('theta_view', value=np.radians(1e-10),          vary=False)
params_3C.add('n1',    value=1,                               vary=False)
params_3C.add('n2',    value=1.33,                            vary=False)
params_3C.add('kappa_0', value=1.0546,                        vary=False)
params_3C.add('zB',    value=100,  min=0.1, max=1000,         vary=False)
params_3C.add('T_W',   value=18,   min=0,   max=40,           vary=False)
params_3C.add('T_W_0', value=20,                              vary=False)
params_3C.add('g_dd',  value=0.02, min=0,   max=10,           vary=True)
params_3C.add('g_dsr', value=1/np.pi, min=0, max=10,          vary=True)
params_3C.add('g_dsa', value=1/np.pi, min=0, max=10,          vary=True)
params_3C.add('d_r',   value=0.01, min=0,   max=0.1,          vary=True)
params_3C.add('fd_d',  value=1,                               vary=False)
params_3C.add('fd_s',  value=1,                               vary=False)
params_3C.add('offset', value=0,  min=0,    max=0.01,         vary=False)

log_params_3C = ['C_0', 'C_Y', 'C_X', 'C_Mie', 'g_dd', 'g_dsr', 'g_dsa', 'd_r']

NOISE      = 0.0031   # sr⁻¹ — update so median chi2_calibrated ≈ (n_obs-n_fit)/n_obs
TILE_SIZE  = 4096
MAX_STEPS  = 100

# SLIC + back-interpolation parameters
N_SEGMENTS  = 5000    # ~0.5% of pixels for a 1000×1000 EnMAP scene
COMPACTNESS = 0.1     # low → segments follow spectral edges
SLIC_SIGMA  = 2.0
K           = 4       # kNN neighbours for IDW back-interpolation
N_COMPONENTS = 6      # PCA components for spectral embedding

## Glint forward model (2D) — identical to NB10

In [ ]:
def forward_glint_2D(fit_param_ds, parameters, wavelengths, pre_3C):
    n2     = float(parameters['n2'].value)
    rho_L  = air_water.fresnel(parameters['theta_view'].value,
                               n1=parameters['n1'].value, n2=n2)
    Ed_d  = np.array(pre_3C['Ed_d'],  dtype=float)
    Ed_sr = np.array(pre_3C['Ed_sr'], dtype=float)
    Ed_sa = np.array(pre_3C['Ed_sa'], dtype=float)
    Ls_Ed = np.array(pre_3C['Ls_Ed'], dtype=float)
    Ed    = Ed_d + Ed_sr + Ed_sa

    def _get(name):
        if name in fit_param_ds:
            return fit_param_ds[name].values
        return fit_param_ds['x_hat'].sel(param=name).values

    g_dd  = _get('g_dd')[...,  np.newaxis]
    g_dsr = _get('g_dsr')[..., np.newaxis]
    g_dsa = _get('g_dsa')[..., np.newaxis]
    d_r   = _get('d_r')[...,   np.newaxis]

    L_s = sky_radiance.L_s(
        float(parameters['fd_d'].value), g_dd, Ed_d,
        float(parameters['fd_s'].value), g_dsr, Ed_sr,
        g_dsa, Ed_sa,
    )
    R_rs_surface  = Rrs_surf(L_s, Ed, rho_L, d_r)
    R_rs_surface += air_water.fresnel(float(parameters['theta_view'].value), n2=n2) * Ls_Ed
    return R_rs_surface.transpose(2, 0, 1)

## Data store

In [ ]:
config = configparser.ConfigParser()
config.read('../../config.ini')
credentials = {k: v.strip() for k, v in config['Credentials'].items()}

store = new_data_store(
    's3', max_depth=5, root='coastal-cubes/sek/',
    storage_options=dict(
        anon=False,
        key=credentials['s3_client_id'],
        secret=credentials['s3_client_secret'],
    )
)

INPUT_PREFIX  = 'helsinki/L2A_land/'
OUTPUT_PREFIX = 'helsinki/temp/'

SCENE_IDS = [
    'ENMAP01-____L2A-DT0000158841_20251019T101424Z_002_V010505_20260206T113145Z',
]

## Run — SLIC superpixel LSQ glint correction

In [ ]:
for scene_id in SCENE_IDS:
    print(f'Processing {scene_id}')

    img         = store.open_data(f'{INPUT_PREFIX}{scene_id}.zarr')
    scene_crs   = img.rio.crs or CRS.from_wkt(img.spatial_ref.attrs['crs_wkt'])
    wavelengths = img.wavelength.values[:80]

    refl  = img['reflectance'].isel(band=slice(0, 80))
    rrs   = (refl.where(refl > -32768) / 10_000) / np.pi
    cloud = (img['cloud'] == 1) | (img['cirrus'] == 1) | (img['haze'] == 1)
    rrs   = rrs.where(~cloud)

    MIN_WATER_PIXELS = 100
    _wc = store.open_data(f'{OUTPUT_PREFIX}{scene_id}-worldcover.zarr')
    _wf = _wc['water_fraction'].values
    _labeled, _ = ndi.label(_wf >= 0.5)
    _sizes = np.bincount(_labeled.ravel())
    _sizes[0] = 0
    ocean_mask = xr.DataArray(
        np.isin(_labeled, np.where(_sizes >= MIN_WATER_PIXELS)[0]),
        coords=_wc['water_fraction'].coords, dims=_wc['water_fraction'].dims,
    )
    rrs = rrs.where(ocean_mask)

    pre_3C   = albert_mobley_3C_jax.precompute(
        wavelengths, fresh=False,
        theta_sun=float(params_3C['theta_sun'].value),
        P=1013.25, AM=1, RH=60, H_oz=0.38, WV=2.5, alpha=1.317, beta=0.2606,
    )
    f_vec_3C = albert_mobley_3C_jax.make_forward_vec(list(params_3C.keys()), pre_3C)

    dummy_sigma_a = {n: 1.0 for n in params_3C if params_3C[n].vary}
    setup_3C = oe_engine.build_inversion(
        params_3C, f_vec_3C, dummy_sigma_a, log_params=log_params_3C
    )

    Rrs_arr = rrs.transpose('y', 'x', 'band').values
    n_rows, n_cols, _ = Rrs_arr.shape
    n_water = int(np.isfinite(Rrs_arr).all(axis=-1).sum())
    print(f'  Scene: {n_rows}×{n_cols}  ({n_water} water pixels)')

    t0 = time.perf_counter()
    results_sp = superpixel_engine.invert_image_superpixel(
        Rrs_arr, setup_3C, NOISE,
        n_segments=N_SEGMENTS,
        compactness=COMPACTNESS,
        sigma=SLIC_SIGMA,
        k=K,
        n_components=N_COMPONENTS,
        invert_fn=lsq_engine.invert_image,
        store_sp_results=True,
        max_steps=MAX_STEPS,
        tile_size=TILE_SIZE,
        use_lm=True,
    )
    t_total = time.perf_counter() - t0

    n_segs = len(results_sp['sp_counts'])
    print(f'  Superpixel LSQ done in {t_total:.1f}s  ({n_segs} segments, {n_water/n_segs:.0f} px/seg)')

    fit_names = results_sp['fit_names']
    coords_2d = {'y': rrs.y, 'x': rrs.x}
    glint_ds = xr.Dataset({
        'x_hat': xr.DataArray(
            results_sp['x_hat'],
            dims=('y', 'x', 'param'),
            coords={**coords_2d, 'param': fit_names},
        ),
        'chi2': xr.DataArray(
            results_sp['chi2'], dims=('y', 'x'), coords=coords_2d,
        ),
        'chi2_calibrated': xr.DataArray(
            results_sp['chi2_calibrated'], dims=('y', 'x'), coords=coords_2d,
        ),
        'n_steps': xr.DataArray(
            results_sp['n_steps'], dims=('y', 'x'), coords=coords_2d,
        ),
    }).rio.write_crs(scene_crs)
    store.write_data(glint_ds,
                     f'{OUTPUT_PREFIX}{scene_id}-glint_params_lsq_sp.zarr', replace=True)

    glint_arr = forward_glint_2D(glint_ds, params_3C, wavelengths, pre_3C)
    glint_da  = xr.DataArray(glint_arr, coords=rrs.coords, dims=rrs.dims).rio.write_crs(scene_crs)
    Rrs_corr  = (rrs - glint_da).rio.write_crs(scene_crs)
    store.write_data(Rrs_corr.to_dataset(name='Rrs'),
                     f'{OUTPUT_PREFIX}{scene_id}-Rrs_lsq_sp.zarr', replace=True)
    print(f'  Saved.')

## Segmentation visualisation

In [ ]:
_labels = results_sp['labels']
_n_segs = len(np.unique(_labels))

# Use bands 2/1/0 as RGB false colour (adjust indices to your wavelengths)
_rgb = Rrs_arr[..., [2, 1, 0]].copy()
_rgb = np.where(np.isfinite(_rgb), _rgb, np.nanmin(_rgb))
_p2, _p98 = np.nanpercentile(_rgb, [2, 98])
_rgb = np.clip((_rgb - _p2) / (_p98 - _p2 + 1e-12), 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(_rgb, origin='upper')
axes[0].set_title('Rrs false colour')
axes[0].axis('off')

_img_bd = mark_boundaries(_rgb, _labels, color=(1, 1, 0), mode='thick')
axes[1].imshow(_img_bd, origin='upper')
axes[1].set_title(f'SLIC segments (n={_n_segs}, compactness={COMPACTNESS}, σ={SLIC_SIGMA})')
axes[1].axis('off')

plt.tight_layout()
plt.show()

sp_c = results_sp['sp_counts']
print(f'Segments: {_n_segs}  |  pixels/seg: min={sp_c.min()}  median={np.median(sp_c):.0f}  max={sp_c.max()}')

## Diagnostics — noise calibration

`chi2_calibrated = chi2_raw × N` adjusts for the smoother mean spectrum.  
For well-calibrated LSQ noise: median `chi2_calibrated` ≈ `(n_obs − n_fit) / n_obs`.

In [ ]:
import hvplot
import holoviews as hv
import hvplot.xarray

_scene = SCENE_IDS[0]
_gp    = store.open_data(f'{OUTPUT_PREFIX}{_scene}-glint_params_lsq_sp.zarr')
_chi2_cal = _gp['chi2_calibrated'].values
_med      = float(np.nanmedian(_chi2_cal[np.isfinite(_chi2_cal)]))

n_obs      = 80
n_fit      = _gp['x_hat'].sizes['param']
chi2_ideal = (n_obs - n_fit) / n_obs
noise_cal  = NOISE * np.sqrt(_med / chi2_ideal)

print(f'NOISE = {NOISE:.5f} sr⁻¹')
print(f'n_obs={n_obs}, n_fit={n_fit}  →  ideal chi2_calibrated = {chi2_ideal:.3f}')
print(f'Median chi2_calibrated = {_med:.3f}  →  calibrated NOISE = {noise_cal:.5f} sr⁻¹')
if abs(_med - chi2_ideal) > 0.05 * chi2_ideal:
    print(f'→ Update NOISE = {noise_cal:.5f} and re-run.')
else:
    print(f'✓ chi2_calibrated ≈ {chi2_ideal:.3f} — noise well-calibrated.')

## Maps

In [ ]:
_scene = SCENE_IDS[0]
_gp    = store.open_data(f'{OUTPUT_PREFIX}{_scene}-glint_params_lsq_sp.zarr')
_valid = np.isfinite(_gp['x_hat']).any('param')
_x_hat    = _gp['x_hat'].where(_valid)
_chi2_cal = _gp['chi2_calibrated'].where(_valid)
_nsteps   = _gp['n_steps'].where(_valid) if 'n_steps' in _gp else None
_opts2 = dict(x='x', y='y', aspect='equal', width=300, height=300, robust=True)

n_obs      = 80
n_fit      = _gp['x_hat'].sizes['param']
chi2_ideal = (n_obs - n_fit) / n_obs

p_chi2 = _chi2_cal.hvplot(
    cmap='RdYlGn_r', clim=(0, 3),
    title=f'chi2_calibrated (target ≈ {chi2_ideal:.2f})', **_opts2,
)

if _nsteps is not None:
    _ns = _nsteps.values
    _ns_valid = _ns[_ns >= 0]
    print(f'mean n_steps: {_ns_valid.mean():.1f}  |  % at max: {100*(_ns_valid == int(_ns_valid.max())).mean():.1f}%')
    p_nsteps = _nsteps.hvplot(cmap='YlOrRd', title='n_steps (superpixel)', **_opts2)
else:
    p_nsteps = None

p_gdd  = _x_hat.sel(param='g_dd').hvplot( cmap='viridis', title='g_dd',  **_opts2)
p_gdsr = _x_hat.sel(param='g_dsr').hvplot(cmap='viridis', title='g_dsr', **_opts2)
p_gdsa = _x_hat.sel(param='g_dsa').hvplot(cmap='viridis', title='g_dsa', **_opts2)
p_dr   = _x_hat.sel(param='d_r').hvplot(  cmap='viridis', title='d_r',   **_opts2)

panels = [p_chi2] + ([p_nsteps] if p_nsteps is not None else []) + [p_gdd, p_gdsr, p_gdsa, p_dr]
hv.Layout(panels).cols(3)

## Comparison — superpixel LSQ vs per-pixel LSQ (NB10)

Load the per-pixel LSQ result from NB10 and compare parameter maps.

In [ ]:
_scene = SCENE_IDS[0]
_gp_px = store.open_data(f'{OUTPUT_PREFIX}{_scene}-glint_params_lsq.zarr')
_gp_sp = store.open_data(f'{OUTPUT_PREFIX}{_scene}-glint_params_lsq_sp.zarr')

_params_to_show = ['g_dd', 'g_dsr', 'g_dsa', 'd_r']
_opts3 = dict(x='x', y='y', aspect='equal', width=280, height=280, robust=True)

panels = []
for pname in _params_to_show:
    _px  = _gp_px['x_hat'].sel(param=pname)
    _sp  = _gp_sp['x_hat'].sel(param=pname)
    _diff = _px - _sp
    panels += [
        _px.hvplot(  title=f'{pname} per-pixel LSQ',  **_opts3),
        _sp.hvplot(  title=f'{pname} superpixel LSQ', **_opts3),
        _diff.hvplot(title=f'{pname} diff (px−sp)',   cmap='RdBu_r', **_opts3),
    ]

hv.Layout(panels).cols(3)

## Glint-corrected Rrs — side-by-side

In [ ]:
_scene = SCENE_IDS[0]
_wl    = store.open_data(f'{INPUT_PREFIX}{_scene}.zarr').wavelength.values[:80]

_Rrs_px = store.open_data(f'{OUTPUT_PREFIX}{_scene}-Rrs_lsq.zarr')['Rrs']
_Rrs_sp = store.open_data(f'{OUTPUT_PREFIX}{_scene}-Rrs_lsq_sp.zarr')['Rrs']
_Rrs_px = _Rrs_px.where(np.isfinite(_Rrs_px).any('band'))
_Rrs_sp = _Rrs_sp.where(np.isfinite(_Rrs_sp).any('band'))

r_idx = int(np.argmin(np.abs(_wl - 670)))
g_idx = int(np.argmin(np.abs(_wl - 550)))
b_idx = int(np.argmin(np.abs(_wl - 460)))
_opts_rgb = dict(x='x', y='y', bands='band', framewise=True, aspect='equal',
                 width=480, height=480, robust=True)

p_px = _Rrs_px.isel(band=[r_idx, g_idx, b_idx]).hvplot.rgb(title='Glint-corrected (per-pixel LSQ)', **_opts_rgb)
p_sp = _Rrs_sp.isel(band=[r_idx, g_idx, b_idx]).hvplot.rgb(title='Glint-corrected (superpixel LSQ)', **_opts_rgb)
(p_px + p_sp)